# Módulo 4 · Notebook 5 — Evaluación de RAG

Un pipeline RAG puede **recuperar mal** o **generar respuestas no fundamentadas**. La evaluación sistemática evita desplegar sistemas que alucinan.

### Métricas clave

| Métrica | Pregunta que responde |
|---------|----------------------|
| **Context precision** | ¿Los chunks recuperados son relevantes? |
| **Faithfulness** | ¿La respuesta se apoya en el contexto? |
| **Answer relevance** | ¿La respuesta contesta la pregunta? |

## 1. Setup — cadena RAG de referencia

In [ ]:
import warnings
from pathlib import Path

from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_ollama import ChatOllama, OllamaEmbeddings

warnings.filterwarnings("ignore")

BASE_DIR = Path("../../").resolve()
VECTOR_DIR = str(BASE_DIR / "data" / "vector_db")

embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")
vectorstore = Chroma(
    persist_directory=VECTOR_DIR,
    embedding_function=embeddings,
    collection_name="ml_papers",
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
llm = ChatOllama(model="llama3.2:1b", temperature=0)
judge_llm = ChatOllama(model="llama3.2:1b", temperature=0)

prompt = PromptTemplate.from_template("""
Responde en español usando SOLO el contexto. Si no hay información suficiente, dilo.

Contexto:
{context}

Pregunta: {question}
Respuesta:""")

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

chain = (
    {
        "context": RunnableLambda(lambda x: x["question"]) | retriever | RunnableLambda(format_docs),
        "question": RunnableLambda(lambda x: x["question"]),
    }
    | prompt | llm | StrOutputParser()
)

def ask(question: str) -> dict:
    docs = retriever.invoke(question)
    context_text = format_docs(docs)
    answer = chain.invoke({"question": question})
    return {
        "question": question,
        "context_text": context_text,
        "answer": answer,
        "sources": docs,
    }

print("✅ Pipeline de evaluación listo")

## 2. Dataset de evaluación

Conjunto pequeño de preguntas con respuesta de referencia (*ground truth*) para comparar.

In [ ]:
EVAL_SET = [
    {
        "question": "¿Cuál es el proceso de ingestión de datos en FinGPT?",
        "ground_truth": "Pipeline con capas de fuentes de datos, ingeniería de datos y LLM financiero.",
    },
    {
        "question": "¿Qué es machine learning?",
        "ground_truth": "Debería indicar que no está en los papers o responder con baja confianza.",
    },
    {
        "question": "¿Qué desafíos tiene el análisis financiero con NLP?",
        "ground_truth": "Alta volatilidad, baja señal/ruido, datos sensibles al tiempo.",
    },
]

print(f"{len(EVAL_SET)} preguntas de evaluación")

## 3. LLM como juez (*LLM-as-judge*)

Pedimos al modelo que puntúe de 1 a 5 tres dimensiones.

In [ ]:
JUDGE_PROMPT = PromptTemplate.from_template("""
Eres un evaluador de sistemas RAG. Puntúa de 1 (malo) a 5 (excelente).
Responde SOLO con tres números separados por comas: contexto, fidelidad, relevancia

Pregunta: {question}
Contexto recuperado: {context}
Respuesta generada: {answer}
Referencia esperada: {ground_truth}

Criterios:
- Contexto: ¿el contexto contiene info útil para la pregunta?
- Fidelidad: ¿la respuesta está apoyada en el contexto (sin inventar)?
- Relevancia: ¿la respuesta contesta la pregunta?

Formato: X,Y,Z""")

def evaluate_item(item: dict, result: dict) -> dict:
    raw = (JUDGE_PROMPT | judge_llm | StrOutputParser()).invoke({
        "question": item["question"],
        "context": result["context_text"][:2000],
        "answer": result["answer"],
        "ground_truth": item["ground_truth"],
    })
    try:
        parts = [int(x.strip()) for x in raw.split(",")[:3]]
        ctx, faith, rel = parts[0], parts[1], parts[2]
    except (ValueError, IndexError):
        ctx = faith = rel = 0
    return {
        "context_score": ctx,
        "faithfulness": faith,
        "relevance": rel,
        "raw_judge": raw.strip(),
    }

print("✅ Juez configurado")

## 4. Ejecutar evaluación

In [ ]:
results = []
for item in EVAL_SET:
    print(f"Evaluando: {item['question'][:60]}...")
    out = ask(item["question"])
    scores = evaluate_item(item, out)
    results.append({**item, **out, **scores})

print(f"\n✅ {len(results)} evaluaciones completadas")

## 5. Resultados

In [ ]:
import pandas as pd

df = pd.DataFrame(results)

avg_ctx = df["context_score"].mean()
avg_faith = df["faithfulness"].mean()
avg_rel = df["relevance"].mean()

print("=== Puntuaciones por pregunta ===")
for r in results:
    print(f"\nQ: {r['question'][:70]}...")
    print(
        f"   Contexto={r['context_score']}  "
        f"Fidelidad={r['faithfulness']}  "
        f"Relevancia={r['relevance']}"
    )
    print(f"   A: {r['answer'][:120]}...")

print(f"\n=== Promedios ===")
print(f"Contexto: {avg_ctx:.2f} | Fidelidad: {avg_faith:.2f} | Relevancia: {avg_rel:.2f}")

## 6. Heurísticas rápidas (sin LLM juez)

Métricas simples que puedes calcular en cualquier entorno.

In [ ]:
def quick_metrics(result: dict) -> dict:
    ctx_len = len(result["context_text"])
    ans_len = len(result["answer"])
    # ¿Palabras del contexto aparecen en la respuesta?
    ctx_words = set(result["context_text"].lower().split())
    ans_words = result["answer"].lower().split()
    overlap = sum(1 for w in ans_words if w in ctx_words and len(w) > 4)
    overlap_ratio = overlap / max(len(ans_words), 1)
    return {
        "context_chars": ctx_len,
        "answer_chars": ans_len,
        "word_overlap_ratio": round(overlap_ratio, 3),
        "num_sources": len(result["sources"]),
    }

for r in results:
    m = quick_metrics(r)
    print(f"Q: {r['question'][:50]}... → overlap={m['word_overlap_ratio']}, sources={m['num_sources']}")

## 7. Resumen y siguientes pasos

- Evalúa **antes de desplegar**: un RAG sin métricas es una caja negra.
- Combina **LLM-as-judge** con heurísticas baratas para iterar rápido.
- En producción considera [RAGAS](https://docs.ragas.io/) o [LangSmith](https://docs.smith.langchain.com/) para trazas y evaluación continua.
- Si faithfulness < 3: reduce temperatura, mejora el prompt o añade "responde solo con el contexto".
- Si context score < 3: prueba MMR, multi-query o re-indexar con otro `chunk_size` (`04_advanced_rag.ipynb`).

**Módulo 4 completado.** Siguiente bloque del curso: **05_agents** — herramientas y agentes ReAct.